# Big Data with DuckDB

<!-- course-learning:start -->
**学习重点**

- [区分磁盘文件体积、表格内存与查询过程的资源使用。](#section-2)
- [直接扫描 Parquet 建立视图，理解视图与整表物化的区别。](#section-3)
- [用 SQL 筛选、分组、计数和去重计数，得到城市汇总。](#section-4)
- [明确关键词比例与每用户平均记录数的分母，并用 Pandas 核对结果。](#section-5-1)
- [先汇总大量输入，再将小结果表交给 Pandas 和 Matplotlib。](#section-6)

**标记说明：** **[重点]** 表示应理解并能做简单修改；**[练习]** 表示动手作答；**[拓展]** 表示进一步学习的方法。未加标记的内容用于准备、解释或查阅。

标记表示学习优先级。代码格依次运行，以建立后续所需的变量；拓展代码格可先观察结果，再深入学习原理。整表内存演示默认关闭；需要启用时，先阅读对应章节的资源说明。
<!-- course-learning:end -->

<!-- course-toc:start -->
<a id="course-toc"></a>
**目录**

- [1 环境准备](#section-1)
- [2 103 MiB 的文件，读入后还是 103 MiB 吗？](#section-2) **[重点]**
  - [2.1 可选：测量完整表的内存](#section-2-1) **[拓展]**
- [3 DuckDB 是什么？怎样直接查询文件？](#section-3) **[重点]**
  - [3.1 核对全国规模与质量条件](#section-3-1) **[重点]**
- [4 SQL 入门：全国哪些城市标签的记录更多？](#section-4) **[重点]**
  - [4.1 同一个文件，如何只分析四川？](#section-4-1) **[重点]**
- [5 从记录量到用户与内容指标](#section-5) **[重点]**
  - [5.1 全国城市表：记录数、不同用户数与“工作”关键词比例](#section-5-1) **[重点]**
  - [5.2 四川：DuckDB 与前课 Pandas 是否得到同一个结果？](#section-5-2) **[重点]**
- [6 汇总后绘图：百万条输入，十行画图数据](#section-6) **[重点]**
- [7 查询计划：引擎实际准备做哪些工作？](#section-7) **[拓展]**
  - [7.1 列裁剪与过滤下推](#section-7-1) **[拓展]**
  - [7.2 看看这一个文件怎样分成行组](#section-7-2) **[拓展]**
- [8 多核与内存：数据继续增大怎么办？](#section-8) **[拓展]**
  - [8.1 多个线程一定比一个线程快吗？](#section-8-1) **[拓展]**
  - [8.2 内存限制和磁盘临时空间](#section-8-2) **[拓展]**
  - [8.3 将两周扩展到全年时，先检查什么？](#section-8-3) **[拓展]**
- [9 附录：DuckDB Spatial](#section-9) **[拓展]**
  - [9.1 安装、加载与创建几何对象](#section-9-1) **[拓展]**
- [10 操作速查](#section-10)
  - [10.1 查阅 API](#section-10-1)
  - [10.2 检查统计口径并释放连接](#section-10-2) **[重点]**
<!-- course-toc:end -->

用 DuckDB 直接查询 Parquet 文件，学习 SQL 筛选与聚合、查询计划、多核计算与内存管理。
重点是在读取和计算过程中减少不需要的列、行和中间结果，再将汇总表交给 Pandas 绘图。

**输入文件：** `tutorials/data/posts_two_weeks.parquet`。
全国窗口为 2022 年 8 月 12–25 日，共 1,254,689 条采集记录、20 列，
Parquet 约 103 MiB；全国可用记录 1,149,693 条。
四川有 54,609 条可用记录，可用于核对 DuckDB 与 Pandas 的计算结果。
“全国”指这个文件内的全国城市标签记录，不表示全平台覆盖或完整用户历史。

<a id="section-1"></a>
## 1 环境准备

在 VS Code 打开整个课程项目，右上角选择课程 Python 内核（Conda 或 venv）。
`tutorials/requirements/environment.yml` 和 `tutorials/requirements/requirements.txt`
都声明了 DuckDB 1.4 LTS。按环境说明选择 Conda 或 uv + venv 安装整套依赖，安装后重启 Notebook 内核。
Conda 的包名为 `python-duckdb`，PyPI 的包名为 `duckdb`；Python 中均使用 `import duckdb`。
Python 原生接口可以直接在 Notebook 中使用；`%%sql` 属于可选的 JupySQL 接口，本教程采用普通 Python cell。
查询 Parquet 不需要网络。运行 Spatial 附录的示例前，需要安装并加载空间扩展。

[官方 Jupyter 指南](https://duckdb.org/docs/lts/guides/python/jupyter)

In [ ]:
# Path 处理文件路径；用 / 拼接目录，不必写本机绝对路径。
from pathlib import Path
# perf_counter 是高精度计时器；两次读数的差值表示经过的秒数。
from time import perf_counter
# gc 管理垃圾回收；删除大表后可请求回收已无引用的对象。
import gc
# os 查询 CPU 等系统信息；它属于 Python 标准库。
import os
# sys 提供当前解释器路径和环境目录，用来检查 Notebook 内核。
import sys
# NumPy 处理数值数组、分位数和布尔条件；np 是常用别名。
import numpy as np
# Pandas 用 Series / DataFrame 处理带行列标签的表格；pd 是别名。
import pandas as pd
# PyArrow 读取 Parquet 的模式、行组和元数据；pq 是别名。
import pyarrow.parquet as pq
# DuckDB 在本机执行 SQL，可直接扫描 Parquet 并只返回查询结果。
import duckdb
# pyplot 创建图形和绘图区，并控制图形显示；plt 是别名。
import matplotlib.pyplot as plt
# font_manager 查询本机字体，便于选择能显示中文的字体。
from matplotlib import font_manager
# PercentFormatter 把 0–1 的比例显示为 0%–100%，不改变原数据。
from matplotlib.ticker import PercentFormatter
# display 在 Notebook 输出区显示表格或图形对象。
from IPython.display import display

# assert 检查当前解释器属于 Conda 或 venv；失败时停止，提示选择安装了课程依赖的内核。
assert (Path(sys.prefix) / 'conda-meta').exists() or sys.prefix != sys.base_prefix, '请选择安装课程依赖的 Conda 或 venv 内核'
# 根据数据所在位置选择相对目录，兼容从 tutorials 或项目根目录启动。
TUTORIAL_DIR = Path('.') if Path('data/posts_two_weeks.parquet').is_file() else Path('tutorials')
# 拼接同一份两周 Parquet 的输入路径，后面的分析都使用它。
DATA = TUTORIAL_DIR / 'data/posts_two_weeks.parquet'
# 确认输入文件存在；路径不正确时先修正启动目录或补齐课堂数据。
assert DATA.is_file(), '请从 tutorials 或项目根目录启动内核，并保留 tutorials/data'
# resolve() 转为完整路径，parent 取得 tutorials 的上一级项目目录。
ROOT = TUTORIAL_DIR.resolve().parent
# 派生表格、图片或临时文件写入此目录，与输入数据分开。
OUT = ROOT / 'data/output/duckdb_tutorial'
# 创建输出目录；parents=True 补建父目录，exist_ok=True 允许目录已存在。
OUT.mkdir(parents=True, exist_ok=True)
# 最多使用 4 个线程；cpu_count() 返回 None 时回退为 1，线程数不等于机器数量。
DEFAULT_THREADS = min(4, os.cpu_count() or 1)
# 默认关闭整表 Pandas 内存演示；改为 True 才执行后面的整表读入分支。
RUN_FULL_MEMORY_DEMO = False  # 选讲：整表读入 Pandas，可能增加约 1 GiB 或更多进程内存。
# 限制表格最多显示的行数，超出部分折叠；不删数据。
pd.set_option('display.max_rows', 10)
# 限制表格最多显示的列数，便于阅读；不删字段。
pd.set_option('display.max_columns', 8)
# 收集本机已安装字体名，不假设每台电脑都有相同的中文字体。
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
# next() 取候选列表中第一个已安装字体；没有匹配时返回 None。
CJK_FONT = next((name for name in ['Noto Sans CJK SC', 'Arial Unicode MS',
    'Microsoft YaHei', 'SimHei', 'Songti SC', 'Heiti TC', 'Noto Sans CJK JP']
    if name in available_fonts), None)
# 仅在找到可用中文字体时设置字体；否则图题使用英文回退。
if CJK_FONT:
    # 把找到的中文字体设为 Matplotlib 默认字体。
    plt.rcParams['font.family'] = CJK_FONT
# 设置后续图形的默认字号、尺寸、负号与边框显示，不改变统计数据。
plt.rcParams.update({'font.size': 11, 'axes.unicode_minus': False,
                     'axes.spines.top': False, 'axes.spines.right': False})

def label(chinese, english):
    """输入：中文、英文字符串；输出：适合当前字体的图题或轴标签。依赖已检查的 CJK_FONT。"""
    # 找到中文字体时返回中文，否则返回英文；函数只选择文字，不改变数据。
    return chinese if CJK_FONT else english

# 打印库版本，便于遇到接口或显示差异时核对环境。
print('DuckDB：', duckdb.__version__, 'Pandas：', pd.__version__)
# 打印内核实际使用的 Python 路径，确认选中了课程环境。
print('Python：', sys.executable)
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('本教程线程上限：', DEFAULT_THREADS)
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('唯一原始数据的相对路径：', DATA)

<a id="size"></a>
<a id="section-2"></a>
## 2 103 MiB 的文件，读入后还是 103 MiB 吗？ **[重点]**

先读 Parquet 元数据确认规模，再用前课的四川范围演示 Pandas 的内存估算。
Parquet 按列存储并压缩；Pandas 中的字符串、索引与对象也占内存，所以文件体积与内存占用不能等同。
**行数、字段类型、文本长度和计算方法一起决定资源需求**。百万行并不自动意味着必须用集群。

`MiB = 字节 / 1024²`；`memory_usage(deep=True)` 尝试计入字符串等对象，
是表内存估算，不是整个 Python 进程的峰值，也不计入后续计算的全部临时对象。

In [ ]:
# 打开 Parquet 元数据接口；查看模式和行组不需要先把全部记录变成 Pandas 表。
parquet = pq.ParquetFile(DATA)
# 从文件状态与 Parquet 元数据读取磁盘体积、行列数和行组数，并格式化为展示表。
file_info = pd.Series({
    '文件字节数': f'{DATA.stat().st_size:,}',  # stat().st_size 是文件在磁盘上的字节数，:, 加千分位。
    '文件大小（MiB）': f'{DATA.stat().st_size / 1024**2:.2f}',  # 1 MiB = 1024² 字节；:.2f 保留两位小数。
    '记录数': f'{parquet.metadata.num_rows:,}',  # 从文件元数据读取总行数，不把正文读入内存。
    '列数': str(parquet.metadata.num_columns),  # 读取模式中的字段数量。
    '行组数': str(parquet.metadata.num_row_groups),  # 行组是 Parquet 存储块，查询可以利用块内列统计跳过无关块。
}, name='实际规模')
# 将 Series 转成单列表格并显示，便于一起查看行标签与统计值。
display(file_info.to_frame())
# 只将四川可用记录的三列读入 Pandas，作为内存演示和 SQL 对照，不另造数据源。
sc_small = pd.read_parquet(DATA, columns=['city', 'user_id', 'text'],
    filters=[('province_code', '==', '51'), ('usable_text', '==', True)])  # 过滤条件为四川省编码 51 且文本可用；同一组条件按“并且”组合。
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('四川可用记录、用户数：', len(sc_small), sc_small['user_id'].nunique())
# deep=True 估算字符串等对象的内存；sum 汇总各列，除以 1024 或 1024² 转为 KiB / MiB。
print('仅这三列的 Pandas 表内存估算（MiB）：',
      round(sc_small.memory_usage(deep=True).sum() / 1024**2, 2))

<a id="section-2-1"></a>
### 2.1 可选：测量完整表的内存 **[拓展]**

`RUN_FULL_MEMORY_DEMO` 默认关闭。测量整表内存时，将其改为 `True`，重新运行初始化单元格和这一格。
完整读取 20 列的内存占用需在当前环境中测量；类型和软件版本会影响结果。
这里读取全部字段，以比较文件体积与表内存；实际查询应先确定需要哪些列。
删除变量可释放对象引用，进程内存不保证立刻全部还给系统。

In [ ]:
# 用 None 表示整表内存尚未测量，避免把未运行误写成 0 MiB。
full_memory_mib = None
# 只有开关为 True 才读入整表；普通课堂路径保持关闭，继续执行 DuckDB 查询。
if RUN_FULL_MEMORY_DEMO:
    # 整表读入 Pandas 会物化所有列和记录，内存通常比压缩后的磁盘文件大。
    full_demo = pd.read_parquet(DATA)
    # deep=True 计入字符串对象；汇总所有列和索引的估算字节数，再转为 MiB。
    full_memory_mib = full_demo.memory_usage(deep=True).sum() / 1024**2
    # 打印计算结果，核对数量、分母、阈值或运行状态。
    print('完整表行列数：', full_demo.shape)
    # 打印计算结果，核对数量、分母、阈值或运行状态。
    print('完整表内存估算（MiB）：', round(full_memory_mib, 2))
    # 打印计算结果，核对数量、分母、阈值或运行状态。
    print('估算与 Parquet 文件体积的比值：',
          round(full_memory_mib / (DATA.stat().st_size / 1024**2), 2))
    # 删除大表变量的引用，让不再使用的内存对象有机会被回收。
    del full_demo
    # 请求垃圾回收；释放对象不保证进程占用内存立即完全归还操作系统。
    gc.collect()
else:
    # 显示指标的口径说明，解释这一步结果应如何理解。
    print('整表内存演示未启用；继续用 DuckDB 查询全国记录。')

<a id="connect"></a>
<a id="section-3"></a>
## 3 DuckDB 是什么？怎样直接查询文件？ **[重点]**

DuckDB 是运行在当前 Python 进程内的分析型数据库引擎。
创建连接后，可以直接查询 Parquet，不需要先运行数据库服务器。
这里的 `:memory:` 表示数据库目录在内存中；源数据仍来自磁盘上的 Parquet，
并不表示要把整份文件全部复制进内存。

`posts` 是一个**视图**：保存“从哪里读数据”的查询定义。建立视图不会另复制一份帖子表。
真正执行查询时仍会消耗读取、解压和计算资源。
`con.execute(...).df()` 则执行 SQL，并把结果转成 Pandas；返回多少行需要由查询控制。

[Python API](https://duckdb.org/docs/lts/clients/python/overview)
与 [Parquet 视图](https://duckdb.org/docs/lts/data/parquet/overview)

In [ ]:
# 若重新运行此格，先关闭之前的连接。
# 重复运行时先检查是否已有连接，关闭旧连接后再新建。
if 'con' in globals():
    # 关闭连接或文件接口，释放相关资源；需要继续查询时重新运行初始化单元格。
    con.close()
# 建立内存数据库连接并设置资源上限；查询溢出可能使用指定的磁盘临时目录。
con = duckdb.connect(database=':memory:', config={
    'threads': DEFAULT_THREADS,  # 允许使用的查询线程上限，不保证每个算子都能并行。
    'memory_limit': '512MB',  # 限制 DuckDB 管理的部分内存；不是整个 Python 进程的硬上限。
    'temp_directory': str(OUT / 'tmp'),  # 可溢写到磁盘的临时目录，不是结果表的持久数据库。
    'max_temp_directory_size': '1GB',  # 限制 DuckDB 临时文件的总空间，还需保证磁盘有空余。
    'autoinstall_known_extensions': False,  # 关闭自动安装扩展，避免普通课堂查询触发下载。
    'autoload_known_extensions': False,  # 关闭自动加载扩展；拓展代码需显式 INSTALL / LOAD。
})
# 把 Parquet 扫描关系命名为 posts 视图；不把整份文件复制成内存中的数据表。
# 重点：视图指向文件中的数据，不先复制整份 Parquet 为 Pandas 大表；查询时按需求扫描。
con.read_parquet(str(DATA)).create_view('posts')
# option_context 临时改变显示行数，离开 with 块后恢复原来的 Pandas 显示设置。
with pd.option_context('display.max_rows', 25):
    # DESCRIBE 返回字段名和类型；df() 转成表格，这里只展示这两列。
    display(con.execute('DESCRIBE posts').df()[['column_name', 'column_type']])

<a id="section-3-1"></a>
### 3.1 核对全国规模与质量条件 **[重点]**

`COUNT(*)` 数行；`COUNT(DISTINCT user_id)` 数不同的非空用户 ID。
`FILTER (WHERE ...)` 只限定它前面的聚合，便于把原始和可用计数放在同一行。
`usable_text` 是已有质量标记，沿用前课规则，不能把不可用记录混进内容比例的分母。

In [ ]:
# execute() 执行 SQL，df() 只把一行汇总结果转为 Pandas，不返回全部帖子。
overview = con.execute("""
    SELECT COUNT(*) AS n_raw,  -- COUNT(*) 统计记录行数；AS 给结果列命名。
           COUNT(*) FILTER (WHERE usable_text) AS n_usable,  -- FILTER 仅筛选这个计数的输入，不会自动过滤其他指标。
           COUNT(DISTINCT user_id) FILTER (WHERE usable_text) AS n_usable_users,  -- COUNT(DISTINCT) 去重数用户；同一组内重复用户只计一次。
           MIN(date) AS start_date, MAX(date) AS end_date,  -- 取得窗口中最早、最晚日期。
           COUNT(DISTINCT date) AS n_dates  -- 去重数日期，得到实际出现记录的不同日期数。
    FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
""").df()
# 在 Notebook 输出区显示本步的结果表。
display(overview)
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert int(overview.loc[0, 'n_raw']) == parquet.metadata.num_rows

**结果解读：** 输出仍只有一行。1,254,689 是原始行数，1,149,693 是可用记录数；
用户数需要在选定质量范围内去重。这里的用户是已观测匿名用户，不是人口规模。
`date` 源字段采用统一的 `YYYY-MM-DD` 字符串，本例可用于核对两周窗口。

<a id="sql"></a>
<a id="section-4"></a>
## 4 SQL 入门：全国哪些城市标签的记录更多？ **[重点]**

从前面的 Pandas 操作理解 SQL：

| SQL | 做什么 | Pandas 中熟悉的操作 |
|---|---|---|
| `FROM posts` | 确定输入 | 选择 DataFrame |
| `WHERE usable_text` | 筛选可用记录 | 布尔条件筛选 |
| `GROUP BY city` | 每个城市一组 | `groupby('city')` |
| `COUNT(*) AS n_posts` | 每组数行并命名 | `size()` |
| `ORDER BY n_posts DESC, city` | 按数量降序，城市名处理并列 | `sort_values` |
| `LIMIT 5` | 限制输出行数 | `head(5)` |

阅读时可按“从哪里读 → 保留哪些行 → 怎样分组 → 算什么 → 排序”理解。
我们先保留完整城市汇总表，再用 `head(10)` 显示前十项，方便后面核对总量。

In [ ]:
# 把城市计数 SQL 保存为字符串，后面可重复执行或加入 EXPLAIN 查看计划。
city_counts_sql = """
    SELECT city, COUNT(*) AS n_posts  -- COUNT(*) 统计记录行数；AS 给结果列命名。
    FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
    WHERE usable_text  -- 过滤记录；WHERE 先于分组与汇总生效。
    GROUP BY city  -- 按城市归组，每个城市返回一行汇总。
    ORDER BY n_posts DESC, city  -- 按指定字段排序；DESC 为降序，第二个排序字段用于打破并列。
"""
# 执行全国城市分组查询，只将城市汇总结果传回 Python。
# 重点：df() 接收的是 SQL 的城市汇总结果；先汇总再传回 Python，可减少传输和表格内存。
national_counts = con.execute(city_counts_sql).df()
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(national_counts.head(10))
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('全国城市汇总行数：', len(national_counts))
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('汇总记录数：', national_counts['n_posts'].sum())
# deep=True 估算字符串等对象的内存；sum 汇总各列，除以 1024 或 1024² 转为 KiB / MiB。
print('汇总表内存估算（KiB）：', round(national_counts.memory_usage(deep=True).sum() / 1024, 2))

**结果解读：** 百万条输入被聚合成每城市一行。城市标签按源字段解释，
数量差异也可能来自采集覆盖；不能据此给城市居民的活跃程度排名。
源数据的 `Unknown` 表示城市未知，不是一座城市：汇总表保留它以核对全部记录，
后面的城市排名图会排除它，并单独报告其记录数。
`LIMIT` 限制的是最终输出；含排序、分组时，返回五行也可能需要先处理大量输入。

<a id="section-4-1"></a>
### 4.1 同一个文件，如何只分析四川？ **[重点]**

`?` 是参数占位符，参数列表按出现顺序传入。省编码是字符串，四川为 `'51'`。
改 `PROVINCE_CODE` 并重跑本格即可查看其他省编码。
参数用于传入值；本例不将它用作表名或列名。

In [ ]:
# 四川的省级编码为字符串 51；参数类型与文件中的编码列一致。
PROVINCE_CODE = '51'
# ? 是值参数占位符；列表按顺序提供参数，避免把输入拼接成 SQL 语法。
province_overview = con.execute("""
    SELECT COUNT(*) AS n_posts, COUNT(DISTINCT user_id) AS n_users  -- COUNT(*) 数帖子行；COUNT(DISTINCT user_id) 在组内去重数用户。
    FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
    WHERE usable_text AND province_code = ?  -- 过滤记录；WHERE 先于分组与汇总生效。
""", [PROVINCE_CODE]).df()
# 在 Notebook 输出区显示本步的结果表。
display(province_overview)
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('省编码：', PROVINCE_CODE)

<a id="metrics"></a>
<a id="section-5"></a>
## 5 从记录量到用户与内容指标 **[重点]**

<a id="section-5-1"></a>
### 5.1 全国城市表：记录数、不同用户数与“工作”关键词比例 **[重点]**

这张表的一行是一个城市标签：

- `n_posts`：可用帖子记录数。
- `n_users`：该城市标签内的不同匿名用户数。
- `n_work`：正文含指定关键词的记录数；`contains` 做字面匹配。
- `work_share`：命中记录数 / 该城市全部可用记录数。
- `posts_per_user`：记录数 / 该城市不同用户数。

`WITH counts AS (...)` 将中间汇总命名，便于下一步计算比例；它本身不要求保存一张磁盘表。
`COALESCE(text, '')` 把缺失正文视为空字符串；`NULLIF(分母, 0)` 避免除零。
同一用户可能出现于多个城市标签，所以各城市用户数相加不一定等于全国去重用户数。

In [ ]:
# 本查询使用字面关键词“工作”；它不等同于用户的职业或真实工作状态。
KEYWORD = '工作'
# WITH 定义临时命名的查询结果，先计数再计算比例，不会创建持久数据表。
# 重点：COUNT(*) 数帖子，COUNT(DISTINCT user_id) 数用户；关键词比例除以本城市帖子数，用户均值除以本城市用户数。
city_metrics_sql = """
    WITH counts AS (  -- 定义仅在本次查询中使用的中间计数结果。
        SELECT city,  -- 指定这一层查询要返回的字段或表达式。
               COUNT(*) AS n_posts,  -- COUNT(*) 统计记录行数；AS 给结果列命名。
               COUNT(DISTINCT user_id) AS n_users,  -- COUNT(DISTINCT) 去重数用户；同一组内重复用户只计一次。
               COUNT(*) FILTER (WHERE contains(COALESCE(text, ''), ?)) AS n_work  -- COALESCE 将缺失正文换为空串；contains 按字面关键词检测，? 由 Python 绑定。
        FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
        WHERE usable_text  -- 过滤记录；WHERE 先于分组与汇总生效。
        GROUP BY city  -- 按城市归组，每个城市返回一行汇总。
    )
    SELECT city, n_posts, n_users, n_work,  -- 指定这一层查询要返回的字段或表达式。
           n_work * 1.0 / NULLIF(n_posts, 0) AS work_share,  -- NULLIF 把零分母变成 NULL；乘 1.0 明确使用小数比例。
           n_posts * 1.0 / NULLIF(n_users, 0) AS posts_per_user  -- NULLIF 把零分母变成 NULL；乘 1.0 明确使用小数比例。
    FROM counts  -- 指定数据视图、中间查询或配置表作为输入。
    ORDER BY n_posts DESC, city  -- 按指定字段排序；DESC 为降序，第二个排序字段用于打破并列。
"""
# 把 KEYWORD 绑定到 SQL 中的 ?，返回每个城市的记录、用户和关键词指标。
national_city = con.execute(city_metrics_sql, [KEYWORD]).df()
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(national_city.head(10))
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('关键词：', KEYWORD)
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('可用记录中的全国不同用户数：', int(overview.loc[0, 'n_usable_users']))
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('各城市用户数之和：', int(national_city['n_users'].sum()))

**结果解读：** 关键词比例描述文字命中，不表示职业、工作状态、偏好或心理状态。
`posts_per_user` 是采集记录的平均数，不能替代中位数，也不能当成完整历史的发帖频率。
更换关键词后，`work_share` 的计算含义随之改变，也需要同步修改图题。

<a id="section-5-2"></a>
### 5.2 四川：DuckDB 与前课 Pandas 是否得到同一个结果？ **[重点]**

这里固定核对四川可用记录，与开场 `sc_small` 的范围一致。
Pandas 已通过 `columns` 和 `filters` 限定读取，所以比较的是两种合理写法。
不用不同质量条件、不同统计单位的结果比较“谁更快”。

In [ ]:
# 使用相同筛选范围和命中规则，在四川子集上核对 SQL 与 Pandas 是否一致。
sc_sql = """
    SELECT city, COUNT(*) AS n_posts, COUNT(DISTINCT user_id) AS n_users,  -- COUNT(*) 数帖子行；COUNT(DISTINCT user_id) 在组内去重数用户。
           COUNT(*) FILTER (WHERE contains(COALESCE(text, ''), ?)) AS n_work  -- COALESCE 将缺失正文换为空串；contains 按字面关键词检测，? 由 Python 绑定。
    FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
    WHERE usable_text AND province_code = '51'  -- 过滤记录；WHERE 先于分组与汇总生效。
    GROUP BY city  -- 按城市归组，每个城市返回一行汇总。
    ORDER BY city  -- 按指定字段排序；DESC 为降序，第二个排序字段用于打破并列。
"""
# 执行四川汇总并按 city 排序；得到与 Pandas 对照表相同的列和行顺序。
sc_duck = con.execute(sc_sql, [KEYWORD]).df()
# 先用相同关键词标记命中；regex=False 按字面匹配，na=False 将缺失文本视为未命中。
sc_pandas = sc_small.assign(has_work=sc_small['text'].str.contains(KEYWORD, regex=False, na=False))
# 按城市聚合计数和关键词命中数；重置索引并按 city 排序，保证与 SQL 表一一对应。
sc_pandas = (sc_pandas.groupby('city', dropna=False)
    .agg(n_posts=('user_id', 'size'), n_users=('user_id', 'nunique'), n_work=('has_work', 'sum'))
    .reset_index().sort_values('city').reset_index(drop=True))
# 逐项核对两个结果表；忽略 dtype 差异，但仍检查行列标签和数值。
pd.testing.assert_frame_equal(sc_duck, sc_pandas, check_dtype=False)
# 显示结果的前几行，观察索引、列名和数值；head 不改变完整统计结果。
display(sc_duck.head(6))
# 显示指标的口径说明，解释这一步结果应如何理解。
print('核对通过：各城市记录数、不同用户数和关键词命中数完全一致。')

<a id="plot"></a>
<a id="section-6"></a>
## 6 汇总后绘图：百万条输入，十行画图数据 **[重点]**

DuckDB 已完成筛选、分组和计数；Pandas 只承接每城市一行的结果。
下面排除城市未知的标签，再取记录最多的十个城市，在同一顺序下比较记录数和关键词比例。
这是按记录量选择的十个城市，**不是关键词比例最高的十个城市**。

如果 `.df()` 接收的是 `SELECT * FROM posts`，仍会把大量原始数据转入 Pandas，
因此关键是先确定最终需要的结果规模。

In [ ]:
# 把缺失城市或 Unknown 标签单列统计，保留这些记录在全国总量中。
unknown_city = national_city.loc[national_city['city'].isna() | national_city['city'].eq('Unknown')]
# 只筛掉未知城市的排名资格，既不删除原始记录，也不从全国总量扣除它们。
known_city = national_city.loc[national_city['city'].notna() & national_city['city'].ne('Unknown')]
# 从已按记录量降序排列的表中取前十城市，再升序排用于横条显示。
top10 = known_city.head(10).sort_values(['n_posts', 'city'])
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('城市未知的可用记录数：', int(unknown_city['n_posts'].sum()), '；只从排名图中排除，保留在总量中。')
# 创建子图（1 行 2 列）；返回 Figure 和 Axes，figsize 为英寸，sharex / sharey 控制共享坐标轴。
fig_city, axes_city = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout='constrained')
# 横向柱形图：类别在纵轴、数值在横轴；返回柱子对象，可用于添加数字标签。
bars = axes_city[0].barh(top10['city'], top10['n_posts'], color='#2878B5')
# 在每根柱子旁标出数量；fmt 控制数字格式，padding 控制文字与柱端的距离。
axes_city[0].bar_label(bars, fmt='%d', padding=3, fontsize=9)
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
axes_city[0].set(title=label('全国记录量前十城市标签', 'Ten city labels with most records'),
    xlabel=label('可用帖子记录数（条）', 'Usable post records'),  # 横轴名称及单位。
    ylabel=label('源数据城市标签', 'Source city label'),  # 纵轴名称及单位。
    xlim=(0, top10['n_posts'].max() * 1.25))
# 横向柱形图：类别在纵轴、数值在横轴；返回柱子对象，可用于添加数字标签。
axes_city[1].barh(top10['city'], top10['work_share'], color='#E07A35')
# 为当前绘图区设置图题、坐标名称和单位；label() 根据字体选择中文或英文文字。
axes_city[1].set(title=label(f'同一组城市：“{KEYWORD}”关键词比例', f'Same cities: keyword "{KEYWORD}" share'),
    xlabel=label('占本城市可用记录的比例', 'Share of usable city records'))  # 横轴名称及单位。
# 调整刻度标签是否显示或旋转角度，使共享坐标的图形更清楚。
axes_city[1].yaxis.set_tick_params(labelleft=False)
# xmax=1 把原值 1 显示为 100%；只改刻度标签，不把数据乘以 100。
axes_city[1].xaxis.set_major_formatter(PercentFormatter(xmax=1))
# 依次操作各个子图；zip 可把绘图区与参数或列名一一配对。
for ax in axes_city:
    # 加浅色网格便于读取数值；axis 指定方向，alpha 控制透明度。
    ax.grid(axis='x', alpha=0.25)
# 在 Notebook 输出区显示当前图形。
plt.show()
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('送入画图的城市数：', len(top10), '；各城市比例的分母为该城市可用记录数。')

<a id="plan"></a>
<a id="section-7"></a>
## 7 查询计划：引擎实际准备做哪些工作？ **[拓展]**

<a id="section-7-1"></a>
### 7.1 列裁剪与过滤下推 **[拓展]**

下面只查一个源文件日期的城市记录数。`source_date` 是采集源文件日期，
此处用于讲文件扫描；不要未经核对就将它当作发帖时间。
`EXPLAIN` 只显示计划，不运行查询；`EXPLAIN ANALYZE` 会真正运行并显示执行统计。

观察 `READ_PARQUET` 或 `TABLE_SCAN` 区域：`Projections` 是需要输出的列，
`Filters` 是交给读取阶段处理的条件；过滤所需列也必须读取。
无需参与此查询的 `text`、位置和情绪分数字段可以跳过。
文件内行组的统计范围合适时，还可能跳过整块数据，效果取决于数据布局。

[Parquet 部分读取](https://duckdb.org/docs/lts/data/parquet/overview)
与 [查询性能指南](https://duckdb.org/docs/lts/guides/performance/how_to_tune_workloads)

In [ ]:
# 查询一个来源日期的可用记录，供计划查看与行组范围分析使用。
plan_sql = """
    SELECT city, COUNT(*) AS n_posts  -- COUNT(*) 统计记录行数；AS 给结果列命名。
    FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
    WHERE usable_text AND source_date = ?  -- 过滤记录；WHERE 先于分组与汇总生效。
    GROUP BY city  -- 按城市归组，每个城市返回一行汇总。
    ORDER BY n_posts DESC, city  -- 按指定字段排序；DESC 为降序，第二个排序字段用于打破并列。
"""
# 指定来源文件日期，供 ? 参数绑定和行组元数据比较共同使用。
SOURCE_DAY = '2022-08-12'
# EXPLAIN 展示计划而不执行实际查询；fetchone() 取第一行，[1] 取计划文本字段。
plan_text = con.execute('EXPLAIN ' + plan_sql, [SOURCE_DAY]).fetchone()[1]
# 打印多行执行计划，观察扫描、过滤、聚合及算子运行信息。
print(plan_text)

In [ ]:
# EXPLAIN ANALYZE 会实际执行查询，并返回本次运行的算子行数与计时信息。
analyzed_plan = con.execute('EXPLAIN ANALYZE ' + plan_sql, [SOURCE_DAY]).fetchone()[1]
# 打印多行执行计划，观察扫描、过滤、聚合及算子运行信息。
print(analyzed_plan)
# 正常执行同一天的城市计数查询，把小结果表传回 Python检查总数。
day_city = con.execute(plan_sql, [SOURCE_DAY]).df()
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('该源文件日期的可用记录总数：', int(day_city['n_posts'].sum()))

<a id="section-7-2"></a>
### 7.2 看看这一个文件怎样分成行组 **[拓展]**

**行组（row group）不是另一份数据文件**，是 Parquet 内部的存储块。
下面只读元数据，展示每个块的行数及源日期最小/最大值。
若某一块的日期范围完全不满足条件，引擎有机会跳过该块。
计划中的 `~... rows` 是估算行数，不能用它替代真实记录数；实际计数以查询结果为准。

In [ ]:
# 找到 source_date 在 Parquet 模式中的列位置，用该位置读取行组统计。
source_date_column = parquet.schema_arrow.names.index('source_date')
# 建立列表，逐行组收集日期范围和记录数，不把记录正文读入 Pandas。
row_groups = []
# 按行组编号遍历元数据；range 从 0 到行组数减 1。
for i in range(parquet.metadata.num_row_groups):
    # 按从 0 开始的编号取一个行组的元数据。
    group = parquet.metadata.row_group(i)
    # 取该行组 source_date 列的统计信息；文件可能没有保存最小值与最大值。
    stats = group.column(source_date_column).statistics
    # 将当前一轮或一个分组的结果字典追加到列表，随后转换为 DataFrame。
    row_groups.append({
        'row_group': i, 'n_rows': group.num_rows,
        'min_source_date': stats.min if stats and stats.has_min_max else None,  # 只有存在最小值统计时才读取，否则记为 None。
        'max_source_date': stats.max if stats and stats.has_min_max else None,  # 没有最大值统计时不能判断日期范围，记为 None。
    })
# 每个字典成为一行，形成行组编号、记录数及来源日期范围表。
row_group_info = pd.DataFrame(row_groups)
# option_context 临时改变显示行数，离开 with 块后恢复原来的 Pandas 显示设置。
with pd.option_context('display.max_rows', 20):
    # 在 Notebook 输出区显示本步的结果表。
    display(row_group_info)
# 范围覆盖 SOURCE_DAY 的行组可能有目标记录；缺统计信息的行组也保留，不能贸然跳过。
possible = row_group_info.loc[
    row_group_info['min_source_date'].isna()
    | ((row_group_info['min_source_date'] <= SOURCE_DAY)
       & (row_group_info['max_source_date'] >= SOURCE_DAY))]
# 打印计算结果，核对数量、分母、阈值或运行状态。
print('仅按日期范围推算的候选行组数：', len(possible), '/', len(row_group_info))
# 显示指标的口径说明，解释这一步结果应如何理解。
print('这是元数据推算，不是磁盘读取字节数或实际跳过块数的测量。')

<a id="resources"></a>
<a id="section-8"></a>
## 8 多核与内存：数据继续增大怎么办？ **[拓展]**

<a id="section-8-1"></a>
### 8.1 多个线程一定比一个线程快吗？ **[拓展]**

DuckDB 可以在一台电脑内并行执行部分查询。这里对同一全国城市指标查询做线程数对比：
每种线程数先运行一次预热，再交替运行三轮，计时包含执行和将小结果表转成 Pandas。
每次都核对结果，报告中位数及范围。

这是**当前机器、当前查询、缓存已预热**的课堂观察，不是跨库或冷磁盘基准。
多核收益取决于扫描规模、存储布局、磁盘、计算量与调度成本；一次结果不能证明普遍加速。
查询里的 `COUNT(DISTINCT ...)` 和关键词匹配也需要计算，工作量不同于只查元数据行数。

In [ ]:
# 用集合去重并排序，对比 1 线程与默认线程数；只有一个选项时无法比较。
thread_options = sorted({1, DEFAULT_THREADS})
# 建立计时结果列表，记录线程数、重复轮次和实际耗时。
timing_records = []
# try / finally 保证查询或检查中途出错时，也会恢复连接的默认线程设置。
try:
    # 依次切换线程数，对同一条 SQL 使用相同参数，保证比较的工作量一致。
    for n_threads in thread_options:
        # 设置当前连接的查询线程数；插入的是程序中的整数，不是外部 SQL 输入。
        con.execute(f'SET threads = {n_threads}')
        # 先运行一次预热，并核对结果；预热耗时不加入正式计时记录。
        warm_result = con.execute(city_metrics_sql, [KEYWORD]).df()
        # 逐项核对两个结果表；忽略 dtype 差异，但仍检查行列标签和数值。
        pd.testing.assert_frame_equal(warm_result, national_city, check_dtype=False)
    # range(1, 4) 执行三轮测量，不包含右端点 4；每轮交替测试顺序。
    for round_number in range(1, 4):
        # 偶数轮反转线程测试顺序，减少先后顺序造成的系统偏差。
        order = thread_options if round_number % 2 else list(reversed(thread_options))
        # 依次切换线程数，对同一条 SQL 使用相同参数，保证比较的工作量一致。
        for n_threads in order:
            # 设置当前连接的查询线程数；插入的是程序中的整数，不是外部 SQL 输入。
            con.execute(f'SET threads = {n_threads}')
            # 在执行查询前记录高精度时刻，计时不包括后面的结果正确性检查。
            start = perf_counter()
            # 实际执行查询并转换为 Pandas；这两步都包含在计时区间中。
            measured = con.execute(city_metrics_sql, [KEYWORD]).df()
            # 查询后时刻减开始时刻得到秒数，再乘 1000 转为毫秒。
            elapsed_ms = (perf_counter() - start) * 1000
            # 逐项核对两个结果表；忽略 dtype 差异，但仍检查行列标签和数值。
            pd.testing.assert_frame_equal(measured, national_city, check_dtype=False)
            # 将当前一轮或一个分组的结果字典追加到列表，随后转换为 DataFrame。
            timing_records.append({'threads': n_threads, 'round': round_number, 'elapsed_ms': elapsed_ms})
finally:
    # 恢复连接的默认线程上限，避免计时实验改变后续查询配置。
    con.execute(f'SET threads = {DEFAULT_THREADS}')
# 把每次测量整理成表，每行对应一种线程设置的一轮查询。
timings = pd.DataFrame(timing_records)
# 按线程数分别计算耗时中位数、最小值和最大值；中位数减少单次波动影响。
timing_summary = timings.groupby('threads')['elapsed_ms'].agg(['median', 'min', 'max'])
# 在 Notebook 输出区显示本步的结果表。
display(timing_summary.rename(columns={'median': '中位数（ms）', 'min': '最小（ms）', 'max': '最大（ms）'}).round(2))
# 线程选项只有一种时说明无法比较，否则计算本查询的一线程 / 多线程中位耗时比。
if len(thread_options) == 1:
    # 显示指标的口径说明，解释这一步结果应如何理解。
    print('本环境只报告一个 CPU，未做不同线程数对比。')
else:
    # 打印计算结果，核对数量、分母、阈值或运行状态。
    print('当前查询的中位耗时比（1线程 / 多线程）：',
          round(timing_summary.loc[1, 'median'] / timing_summary.loc[DEFAULT_THREADS, 'median'], 2))

<a id="section-8-2"></a>
### 8.2 内存限制和磁盘临时空间 **[拓展]**

开场连接设置了 `memory_limit='512MB'`、线程上限和课程输出目录内的临时路径。
部分排序、分组和连接在内存不足时，可以将中间数据暂存磁盘，称为**溢写（spill）**。
用磁盘换内存需要额外 I/O、空间和时间，也有算子限制；不能保证所有查询都能在任意小内存中运行。
`memory_limit` 管理的内存也不等于整个 Python 进程的硬上限，Pandas 结果等另占内存。

当前样例不一定触发溢写。下方只查看设置，不把设置成功当成超内存计算的验证。
`temp_directory` 是临时计算路径，不是新增原始数据集；上限 1GB 指临时磁盘空间的配置值。

In [ ]:
# 从 duckdb_settings() 查询当前连接的实际配置，用来核对设置是否生效。
settings = con.execute("""
    SELECT name, value  -- 指定这一层查询要返回的字段或表达式。
    FROM duckdb_settings()  -- 指定数据视图、中间查询或配置表作为输入。
    WHERE name IN ('threads', 'memory_limit', 'temp_directory', 'max_temp_directory_size')  -- 过滤记录；WHERE 先于分组与汇总生效。
    ORDER BY name  -- 按指定字段排序；DESC 为降序，第二个排序字段用于打破并列。
""").df()
# 在 Notebook 输出区显示本步的结果表。
display(settings)

<a id="section-8-3"></a>
### 8.3 将两周扩展到全年时，先检查什么？ **[拓展]**

1. **字段和质量口径一致吗？** 先明确日期、用户去重和可用文本条件。
2. **查询需要原始全文吗？** 城市计数无需读正文，关键词比例才需要正文。
3. **能否先减少结果？** 在引擎内筛选和聚合，再将小表交给 Pandas。
4. **数据怎样组织？** 更大的项目可按常用筛选字段组织 Parquet；读取多文件时仍可用同一 SQL 思路。
5. **瓶颈在哪里？** 用执行计划和实际耗时检查，再调整线程、内存与临时磁盘空间。

如果进行按块统计，记录数等计数容易合并；跨块不同用户数不能把各块去重数简单相加。
**原始大表交给 DuckDB 计算，最终分析结果交给 Pandas 整理和绘图。**

<a id="spatial"></a>
<a id="section-9"></a>
## 9 附录：DuckDB Spatial **[拓展]**

输入文件包含 `longitude_grid`、`latitude_grid` 和 `coordinate_valid` 字段。
`spatial` 扩展提供几何创建、空间关系、距离与空间文件读取等能力。

| 函数入口 | 后续能做什么 | 使用前需要明确 |
|---|---|---|
| `ST_Point(x, y)` | 从坐标生成点 | 横纵坐标顺序、坐标来源 |
| `ST_Intersects(a, b)` | 判断几何是否相交 | 几何对象必须采用兼容坐标体系 |
| `ST_Distance(a, b)` | 计算平面几何距离 | 结果单位来自坐标；经纬度直接计算不能称为米 |
| `ST_Read(...)` | 读取空间文件 | 格式支持、坐标体系与字段 |

<a id="section-9-1"></a>
### 9.1 安装、加载与创建几何对象 **[拓展]**

扩展首次安装需要可用网络与匹配的 DuckDB 版本、平台；安装后在连接中显式加载。
`INSTALL` 只准备扩展，`LOAD` 使当前连接可以调用它。

```python
# 首次安装 Spatial 扩展需要联网下载；属于拓展示例，不是默认执行步骤。
con.execute('INSTALL spatial')  # 首次使用时安装扩展
# 在当前连接加载已安装的 Spatial 扩展，让 ST_* 空间函数可用。
con.execute('LOAD spatial')
# 查询少量网格中心点的 WKT 表示；构造几何对象不等于转换坐标系。
spatial_preview = con.execute("""
    SELECT ST_AsText(ST_Point(longitude_grid, latitude_grid)) AS point_wkt  -- 用经度、纬度构造点并转成 WKT 文本；不执行坐标变换。
    FROM posts  -- 指定数据视图、中间查询或配置表作为输入。
    WHERE usable_text AND coordinate_valid  -- 过滤记录；WHERE 先于分组与汇总生效。
      AND province_code = '51'  -- 与上面的条件同时满足；这里只取四川省编码 51。
      AND longitude_grid IS NOT NULL AND latitude_grid IS NOT NULL  -- 排除坐标缺失记录，避免构造无效几何点。
    LIMIT 3  -- 只返回前三行示例，避免展示逐帖位置数据。
""").df()
# 在 Notebook 输出区显示本步的结果表。
display(spatial_preview)
```

这些点来自源数据舍入后的 0.02 度网格，不是精确个人位置；
`coordinate_valid` 只是粗范围质量标记，坐标体系和定位机制尚未核实。
创建几何对象不会解决这些不确定性，不能立即据此计算通勤距离或恢复路线。
确认坐标体系之后，再设计行政区空间连接和距离分析。

[Spatial 扩展安装与加载](https://duckdb.org/docs/lts/core_extensions/spatial/overview)
与 [空间函数参考](https://duckdb.org/docs/lts/core_extensions/spatial/functions)

<a id="reference"></a>
<a id="section-10"></a>
## 10 操作速查

| 目的 | 本教程入口 |
|---|---|
| 直接指向 Parquet | `con.read_parquet(str(DATA)).create_view('posts')` |
| 运行 SQL 并接收小表 | `con.execute(sql, parameters).df()` |
| 数记录 / 数不同用户 | `COUNT(*)` / `COUNT(DISTINCT user_id)` |
| 筛选可用范围 | `WHERE usable_text AND province_code = ?` |
| 关键词字面匹配 | `contains(COALESCE(text, ''), ?)` |
| 看计划 / 实际执行统计 | `EXPLAIN` / `EXPLAIN ANALYZE` |
| 控制线程数 | `SET threads = 1` |
| 控制计算资源 | 连接配置 `memory_limit`、`temp_directory` |

<a id="section-10-1"></a>
### 10.1 查阅 API

[Python API](https://duckdb.org/docs/lts/clients/python/overview)、
[参数化查询](https://duckdb.org/docs/lts/clients/python/dbapi)、
[Parquet 读取](https://duckdb.org/docs/lts/data/parquet/overview)、
[性能与内存](https://duckdb.org/docs/lts/guides/performance/how_to_tune_workloads)。

<a id="section-10-2"></a>
### 10.2 检查统计口径并释放连接 **[重点]**

下面核对城市计数与总量、关键词比例范围，以及四川的统一范围。
关闭连接后若要重跑 SQL，先重跑第 2 部分的连接单元。

In [ ]:
# 逐项核对两个结果表；忽略 dtype 差异，但仍检查行列标签和数值。
pd.testing.assert_frame_equal(national_counts, national_city[['city', 'n_posts']], check_dtype=False)
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert national_city['n_posts'].sum() == int(overview.loc[0, 'n_usable'])
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert national_city['n_work'].between(0, national_city['n_posts']).all()
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert national_city['work_share'].between(0, 1).all()
# 核对两种计算的数值是否在浮点容差内一致，避免直接比较小数舍入误差。
np.testing.assert_allclose(national_city['work_share'], national_city['n_work'] / national_city['n_posts'])
# 核对两种计算的数值是否在浮点容差内一致，避免直接比较小数舍入误差。
np.testing.assert_allclose(national_city['posts_per_user'], national_city['n_posts'] / national_city['n_users'])
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert sc_duck['n_posts'].sum() == len(sc_small) == 54609
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert sc_small['user_id'].nunique() == 36463
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert known_city['n_posts'].sum() + unknown_city['n_posts'].sum() == int(overview.loc[0, 'n_usable'])
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert top10['city'].notna().all() and top10['city'].ne('Unknown').all()
# 用 assert 核对计数、分母或指标范围；不符合预期时抛出异常，不会自动修改结果。
assert row_group_info['n_rows'].sum() == parquet.metadata.num_rows
# 显示指标的口径说明，解释这一步结果应如何理解。
print('全国总量、指标分母和四川 Pandas 对照核对通过；本课只使用一份原始 Parquet。')
# 关闭连接或文件接口，释放相关资源；需要继续查询时重新运行初始化单元格。
con.close()
# 关闭连接或文件接口，释放相关资源；需要继续查询时重新运行初始化单元格。
parquet.close()